# Debug Notebook (Self-Contained)

Embedded code for SDF/FC1 and Policy/Value, with tests.


In [5]:
# --- Begin embedded: config/constants.py ---
"""
配置常量模块
定义所有模型和训练所需的常量参数
"""

from enum import IntEnum
from typing import List, Tuple
import torch


class SIMMODEL(IntEnum):
    """
    Firm-state向量的列顺序枚举
    必须与训练时的输入格式完全一致
    s_i = (b, z, η, i, x, ĉf, ln Kf)
    """
    B = 0      # 杠杆/债务状态 (firm-level)
    Z = 1      # 生产率/冲击 (firm-level)
    ETA = 2    # 是否可再融资 (firm-level, {0,1})
    I = 3      # 投资成本/摩擦 (firm-level, continuous)
    X = 4      # 宏观状态 (macro-level)
    HATCF = 5  # 宏观消费相关状态 (macro-level)
    LNKF = 6   # 宏观资本对数 (macro-level)


class Config:
    """
    全局配置类
    包含所有模型、数据生成和训练所需的参数
    """
    
    # ========== 经济参数 ==========
    # AR(1) 宏观冲击参数
    RHO_X = 0.95          # 宏观生产率自相关系数
    SIGMA_X = 0.012       # 宏观冲击标准差
    XBAR = -2.0           # 宏观生产率均值
    
    # AR(1) 公司冲击参数
    RHO_Z = 0.90          # 公司生产率自相关系数
    SIGMA_Z = 0.36        # 公司冲击标准差
    ZBAR = 0.0            # 公司生产率均值
    
    # 折旧与增长
    DELTA = 0.02          # 折旧率
    G = 1.14              # 增长因子
    
    # 税率与成本
    TAU = 0.2             # 公司税率
    KAPPA_B = 0.004       # 债务融资成本
    KAPPA_E = 0.025       # 股权融资成本
    PHI = 0.4             # 破产成本参数
    
    # 投资与再融资
    I_THRESHOLD = 0.5     # 投资成本上限
    ZETA = 0.03           # 再融资可得性概率
    
    # SDF 参数
    BETA = 0.942          # 贴现因子
    GAMMA = 4.0           # 风险厌恶系数
    SIGMA = 2.0           # 消费波动系数（与 DL7 一致）
    KAPPA = -6.0          # 价值函数幂次 (1-GAMMA)/(1-1/SIGMA)
    SIGMA_C = 0.01        # 消费波动系数
    
    # ========== 模型架构参数 ==========
    # 共享层维度
    SHARE_LAYER_HIDDEN_DIMS = [128, 128, 64]
    
    # 各 Head 维度
    Q_HEAD_DIMS = [32, 16]
    BP0_HEAD_DIMS = [32, 16]
    BPI_HEAD_DIMS = [32, 16]
    P0_HEAD_DIMS = [32, 16]
    PI_HEAD_DIMS = [32, 16]
    
    # SDF & FC1 维度
    SDF_HIDDEN_DIMS = [64, 32]
    FC1_INPUT_DIM = 4      # (x_{t-1}, x_t, ĉf_{t-1}, ln Kf_{t-1})
    FC1_HIDDEN_DIMS = [32, 16]
    
    # FC2 维度
    FC2_INPUT_DIM = 201    # 100(b分位点) + 100(z分位点) + x
    FC2_HIDDEN_DIMS = [128, 64, 32]
    FC2_OUTPUT_DIM = 2     # (ĉ, ln K)
    
    # ========== 数据参数 ==========
    GROUP_SIZE = 2         # 每条path的公司数（sample模式）
    SIMULATE_GROUP_SIZE = 200  # 每条path的公司数（simulate模式）
    BRANCH_NUM = 2         # 分支数
    QUANTILE_NUM = 100     # 分位数数量
    
    # ========== 训练参数 ==========
    BATCH_SIZE = 512
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-3
    MAX_GRAD_NORM = 1.0
    
    # Loss 权重
    LAMBDA_SDF = 1.0
    LAMBDA_FC = 1.0
    LAMBDA_TRANS = 0.5     # FC2 跨期一致性权重
    
    # AIO 权重（动态残差插值）
    AIO_WEIGHT = 0.5
    
    # Z 值惩罚参数
    ALPHA_Z = 1.0
    BETA_Z = 5.0
    Z0 = 1.0
    
    # SDF 矩约束
    MU_LO = -0.025
    MU_HI = 0.0
    VAR_HI = 0.25
    
    # ========== 设备配置 ==========
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    @classmethod
    def firm_state_dim(cls) -> int:
        """返回firm-state向量维度"""
        return 7
    
    @classmethod
    def base_state_dim(cls) -> int:
        """返回base-state向量维度（不含投资成本i）"""
        return 6
    
    @classmethod
    def policy_model_input_var(cls) -> List[int]:
        """返回Policy模型输入的变量索引（不含i）"""
        return [SIMMODEL.B, SIMMODEL.Z, SIMMODEL.ETA, 
                SIMMODEL.X, SIMMODEL.HATCF, SIMMODEL.LNKF]
    
    @classmethod
    def get_ar1_stationary_var(cls, rho: float, sigma: float) -> float:
        """计算AR(1)稳态方差"""
        return sigma**2 / (1 - rho**2)
    
    @classmethod
    def z_stationary_std(cls) -> float:
        """返回z的稳态标准差"""
        return (cls.SIGMA_Z**2 / (1 - cls.RHO_Z**2)) ** 0.5
    
    @classmethod
    def x_stationary_std(cls) -> float:
        """返回x的稳态标准差"""
        return (cls.SIGMA_X**2 / (1 - cls.RHO_X**2)) ** 0.5
# --- End embedded: config/constants.py ---


In [6]:
# --- Begin embedded: models/base.py ---
"""
基础模型组件
包含 MLP 基类和带标准化功能的 MLP
"""

import torch
import torch.nn as nn
from typing import List, Optional, Tuple
import numpy as np


class MLP(nn.Module):
    """
    基础多层感知机
    支持灵活的隐藏层配置和激活函数选择
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int],
        output_dim: int,
        activation: str = 'relu',
        output_activation: Optional[str] = None,
        dropout: float = 0.0,
        use_batch_norm: bool = False
    ):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        
        # 构建网络层
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            
            layers.append(self._get_activation(activation))
            
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_dim = hidden_dim
        
        # 输出层
        layers.append(nn.Linear(prev_dim, output_dim))
        
        if output_activation:
            layers.append(self._get_activation(output_activation))
        
        self.network = nn.Sequential(*layers)
        
        # 初始化权重
        self._init_weights()
    
    def _get_activation(self, activation: str) -> nn.Module:
        """获取激活函数"""
        activations = {
            'relu': nn.ReLU(),
            'leaky_relu': nn.LeakyReLU(0.1),
            'elu': nn.ELU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid(),
            'softplus': nn.Softplus(),
            'gelu': nn.GELU(),
        }
        return activations.get(activation, nn.ReLU())
    
    def _init_weights(self):
        """Xavier 初始化"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


class MLPWithScaler(MLP):
    """
    带标准化功能的 MLP
    用于 FC1 等需要标准化输出的模型
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int],
        output_dim: int = 1,
        **kwargs
    ):
        super().__init__(input_dim, hidden_dims, output_dim, **kwargs)
        
        # 注册标准化参数为 buffer（不参与梯度更新但会被保存）
        self.register_buffer('y_mean_', torch.zeros(output_dim))
        self.register_buffer('y_std_', torch.ones(output_dim))
        self.register_buffer('x_mean_', torch.zeros(input_dim))
        self.register_buffer('x_std_', torch.ones(input_dim))
        self.register_buffer('fitted_', torch.tensor(False))
    
    def fit_scaler(
        self, 
        x: torch.Tensor, 
        y: Optional[torch.Tensor] = None,
        fit_x: bool = True,
        fit_y: bool = True
    ):
        """
        拟合标准化参数
        
        Args:
            x: 输入数据
            y: 输出数据（用于目标标准化）
            fit_x: 是否拟合输入标准化
            fit_y: 是否拟合输出标准化
        """
        if fit_x:
            self.x_mean_ = x.mean(dim=0)
            self.x_std_ = x.std(dim=0).clamp(min=1e-6)
        
        if fit_y and y is not None:
            self.y_mean_ = y.mean(dim=0) if y.dim() > 1 else y.mean().unsqueeze(0)
            self.y_std_ = y.std(dim=0).clamp(min=1e-6) if y.dim() > 1 else y.std().clamp(min=1e-6).unsqueeze(0)
        
        self.fitted_ = torch.tensor(True)
    
    def transform_x(self, x: torch.Tensor) -> torch.Tensor:
        """标准化输入"""
        # 确保输入与 scaler buffer 保持同一 dtype，避免 float/double 混用
        x = x.to(self.x_mean_.dtype)
        return (x - self.x_mean_) / self.x_std_
    
    def transform_y(self, y: torch.Tensor) -> torch.Tensor:
        """标准化输出目标"""
        return (y - self.y_mean_) / self.y_std_
    
    def inverse_transform(self, y_norm: torch.Tensor) -> torch.Tensor:
        """反标准化输出"""
        return y_norm * self.y_std_ + self.y_mean_
    
    def forward_normalized(self, x: torch.Tensor) -> torch.Tensor:
        """
        标准化前向传播
        输入会被标准化，输出保持在标准化空间
        """
        x_norm = self.transform_x(x)
        return super().forward(x_norm)
    
    def forward_physical(self, x: torch.Tensor) -> torch.Tensor:
        """
        物理尺度前向传播
        输入被标准化，输出被反标准化回物理尺度
        """
        y_norm = self.forward_normalized(x)
        return self.inverse_transform(y_norm)


class ResidualBlock(nn.Module):
    """残差块"""
    
    def __init__(self, dim: int, activation: str = 'relu', dropout: float = 0.0):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            self._get_activation(activation),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.activation = self._get_activation(activation)
    
    def _get_activation(self, activation: str) -> nn.Module:
        activations = {
            'relu': nn.ReLU(),
            'leaky_relu': nn.LeakyReLU(0.1),
            'elu': nn.ELU(),
            'gelu': nn.GELU(),
        }
        return activations.get(activation, nn.ReLU())
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(x + self.block(x))


class AttentionMLP(nn.Module):
    """
    带自注意力机制的 MLP
    用于处理变长序列或需要关注不同特征的场景
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int],
        output_dim: int,
        num_heads: int = 4,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # 输入投影
        self.input_proj = nn.Linear(input_dim, hidden_dims[0])
        
        # 自注意力层
        self.attention = nn.MultiheadAttention(
            hidden_dims[0], 
            num_heads, 
            dropout=dropout,
            batch_first=True
        )
        
        # MLP 部分
        layers = []
        prev_dim = hidden_dims[0]
        for hidden_dim in hidden_dims[1:]:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 投影
        h = self.input_proj(x)
        
        # 添加序列维度用于注意力
        h = h.unsqueeze(1)
        h_attn, _ = self.attention(h, h, h)
        h = h + h_attn
        h = h.squeeze(1)
        
        # MLP
        return self.mlp(h)
# --- End embedded: models/base.py ---


In [7]:
# --- Begin embedded: losses/utils.py ---
"""
损失函数工具函数

包含各 Loss 共用的工具函数
"""

import torch
import torch.nn.functional as F
from typing import Tuple


def softplus_gate(x: torch.Tensor, delta: float = 1e-3) -> torch.Tensor:
    """
    平滑阶跃函数（softplus gate）
    
    用于在约束边界处提供平滑的过渡。
    gate(x) = softplus(x/δ) / (softplus(x/δ) + softplus(-x/δ))
    
    Args:
        x: 输入张量
        delta: 平滑度参数，控制过渡的陡峭程度。delta 越小，过渡越陡峭。
    
    Returns:
        gate: 平滑的阶跃输出，范围在 [0, 1] 之间
    """
    z = x / delta
    softplus_pos = F.softplus(z)
    softplus_neg = F.softplus(-z)
    gate = softplus_pos / (softplus_pos + softplus_neg + 1e-8)
    return gate


# 别名，保持向后兼容
smooth_step = softplus_gate


def compute_z_penalty(
    residual: torch.Tensor,
    z: torch.Tensor,
    alpha: float = 1.0,
    beta: float = 5.0,
    z0: float = 1.0
) -> torch.Tensor:
    """
    计算 z 值惩罚
    
    penalty = α * mean(sigmoid(β(z - z0)) * residual)
    
    对高 z 值区域的残差给予更高权重
    
    Args:
        residual: 残差张量
        z: z 值张量
        alpha: 惩罚权重
        beta: sigmoid 斜率
        z0: z 的阈值
    
    Returns:
        z 惩罚值
    """
    weight = torch.sigmoid(beta * (z - z0))
    penalty = alpha * (weight * residual).mean()
    return penalty


def compute_monotonicity_penalty(
    value: torch.Tensor,
    inputs: torch.Tensor,
    input_idx: int,
    direction: str = 'negative'
) -> torch.Tensor:
    """
    计算单调性惩罚
    
    通过梯度约束确保函数对某个输入的单调性
    
    Args:
        value: 函数输出值
        inputs: 输入张量（需要开启 requires_grad）
        input_idx: 要检查单调性的输入维度索引
        direction: 'negative'（应单调递减）或 'positive'（应单调递增）
    
    Returns:
        单调性惩罚
    """
    # 计算梯度
    grad = torch.autograd.grad(
        outputs=value.sum(),
        inputs=inputs,
        create_graph=True,
        retain_graph=True
    )[0]
    
    # 取指定维度的梯度
    grad_dim = grad[:, input_idx]
    
    if direction == 'negative':
        # 应该单调递减：惩罚正梯度
        penalty = F.relu(grad_dim).mean()
    else:
        # 应该单调递增：惩罚负梯度
        penalty = F.relu(-grad_dim).mean()
    
    return penalty


def compute_aio_residual(
    losses: list,
    aio_weight: float = 0.5
) -> torch.Tensor:
    """
    计算动态残差插值（AIO）- 支持任意数量的路径
    
    对于 N 条路径：
    - stable = (1/N) * Σ_j loss_j²
    - aio = |Π_j loss_j|
    residual = (1 - w) * stable + w * aio
    
    Args:
        losses: List[torch.Tensor] - 各路径的损失列表
        aio_weight: AIO 权重
    
    Returns:
        组合后的残差
    """
    if not isinstance(losses, (list, tuple)):
        # 向后兼容：如果只传入两个参数
        raise ValueError("losses must be a list of tensors")
    
    n_paths = len(losses)
    
    # stable 部分：平均平方损失
    residual_stable = sum(loss.pow(2) for loss in losses) / n_paths
    
    # AIO 部分：所有路径损失的乘积
    residual_aio = losses[0]
    for loss in losses[1:]:
        residual_aio = residual_aio * loss
    residual_aio = residual_aio.abs()
    
    return (1 - aio_weight) * residual_stable + aio_weight * residual_aio


def compute_aio_residual_legacy(
    loss1: torch.Tensor,
    loss2: torch.Tensor,
    aio_weight: float = 0.5
) -> torch.Tensor:
    """
    计算动态残差插值（AIO）- 旧版双路径接口
    
    residual = (1 - w) * stable + w * aio
    其中：
    - stable = 0.5 * (loss1² + loss2²)
    - aio = |loss1 * loss2|
    
    Args:
        loss1: 路径1的损失
        loss2: 路径2的损失
        aio_weight: AIO 权重
    
    Returns:
        组合后的残差
    """
    return compute_aio_residual([loss1, loss2], aio_weight)


def compute_moment_penalty(
    M: torch.Tensor,
    mu_lo: float = -0.025,
    mu_hi: float = 0.0,
    var_hi: float = 0.25,
    eps: float = 1e-6
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    计算 SDF 的矩约束惩罚
    
    一阶矩：log(E[M]) ∈ [μ_lo, μ_hi]
    二阶矩：log(Var[M]) ≤ var_hi
    
    Args:
        M: SDF 张量
        mu_lo: 一阶矩下界
        mu_hi: 一阶矩上界
        var_hi: 二阶矩上界
        eps: 数值稳定性常数
    
    Returns:
        L1: 一阶矩惩罚
        L2: 二阶矩惩罚
    """
    # 一阶矩
    mu = M.mean()
    log_mu = torch.log(mu + eps)
    
    # 一阶矩惩罚
    L_low = smooth_step(mu_lo - log_mu) * 100 * (log_mu - mu_lo).pow(2)
    L_high = smooth_step(log_mu - mu_hi) * 100 * (log_mu - mu_hi).pow(2)
    L1 = L_low + L_high
    
    # 二阶矩
    var = M.pow(2).mean() - mu.pow(2)
    log_var = torch.log(var + eps)
    
    # 二阶矩惩罚
    L2 = smooth_step(log_var - var_hi) * 100 * (log_var - var_hi).pow(2)
    
    return L1, L2


def compute_cashflow(
    x: torch.Tensor,
    z: torch.Tensor,
    b: torch.Tensor,
    delta: float,
    tau: float
) -> torch.Tensor:
    """
    计算税后利润
    
    prod = (exp(x+z) - δ - b) - τ * max(exp(x+z) - δ - b, 0)
    
    Args:
        x: 宏观生产率
        z: 公司生产率
        b: 杠杆
        delta: 折旧率
        tau: 税率
    
    Returns:
        税后利润
    """
    profit = torch.exp(x + z) - delta - b
    prod = profit - tau * F.relu(profit)
    return prod
# --- End embedded: losses/utils.py ---


In [8]:
# --- Begin embedded: models/share_layer.py ---
"""
ShareLayer 模型
Policy & Value 的统一架构：共享特征层 + 多头输出

关键结构约束：
- Base state (不含i): (b, z, η, x, ĉ, ln K) ∈ ℝ^6
- Investment cost: i ∈ ℝ
- No-invest heads (不吃i): (Q, bp⁰, P⁰)
- Invest heads (吃i): (bpᴵ, Pᴵ)
"""

import torch
import torch.nn as nn
from typing import Dict, Tuple, List, Optional



class ShareLayer(nn.Module):
    """
    共享特征层
    
    输入：base_state = (b, z, η, x, ĉ, ln K)
    输出：共享表征 h
    """
    
    def __init__(
        self,
        input_dim: int = 6,
        hidden_dims: List[int] = None,
        output_dim: int = 64,
        activation: str = 'relu',
        dropout: float = 0.0
    ):
        super().__init__()
        
        if hidden_dims is None:
            hidden_dims = Config.SHARE_LAYER_HIDDEN_DIMS
        
        self.network = MLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=output_dim,
            activation=activation,
            dropout=dropout
        )
        
        self.output_dim = output_dim
    
    def forward(self, base_state: torch.Tensor) -> torch.Tensor:
        """
        Args:
            base_state: (batch, 6) - (b, z, η, x, ĉ, ln K)
        Returns:
            h: (batch, output_dim) - 共享表征
        """
        return self.network(base_state)


class QHead(nn.Module):
    """
    债券价值 Q 的输出头
    不依赖投资成本 i
    """
    
    def __init__(
        self,
        input_dim: int = 64,
        hidden_dims: List[int] = None,
        output_activation: str = 'softplus'
    ):
        super().__init__()
        
        if hidden_dims is None:
            hidden_dims = Config.Q_HEAD_DIMS
        
        self.network = MLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation=output_activation
        )
    
    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """
        Args:
            h: (batch, input_dim) - 共享表征
        Returns:
            Q: (batch, 1) - 债券价值
        """
        return self.network(h)


class BpHead(nn.Module):
    """
    杠杆候选 bp 的输出头
    bp⁰ 不依赖 i，bpᴵ 依赖 i
    """
    
    def __init__(
        self,
        input_dim: int = 64,
        hidden_dims: List[int] = None,
        requires_i: bool = False,
        output_activation: str = 'sigmoid'  # 约束 bp ∈ [0, 1]
    ):
        super().__init__()
        
        self.requires_i = requires_i
        actual_input_dim = input_dim + 1 if requires_i else input_dim
        
        if hidden_dims is None:
            hidden_dims = Config.BP0_HEAD_DIMS if not requires_i else Config.BPI_HEAD_DIMS
        
        self.network = MLP(
            input_dim=actual_input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation=output_activation
        )
    
    def forward(self, h: torch.Tensor, i: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            h: (batch, input_dim) - 共享表征
            i: (batch, 1) - 投资成本（仅当 requires_i=True 时使用）
        Returns:
            bp: (batch, 1) - 杠杆候选
        """
        if self.requires_i:
            if i is None:
                raise ValueError("Investment cost i is required for bpI head")
            h = torch.cat([h, i], dim=-1)
        
        return self.network(h)


class PHead(nn.Module):
    """
    股票价值 P 的输出头
    P⁰ 不依赖 i，Pᴵ 依赖 i
    """
    
    def __init__(
        self,
        input_dim: int = 64,
        hidden_dims: List[int] = None,
        requires_i: bool = False,
        output_activation: str = 'softplus'  # 保证 P > 0
    ):
        super().__init__()
        
        self.requires_i = requires_i
        actual_input_dim = input_dim + 1 if requires_i else input_dim
        
        if hidden_dims is None:
            hidden_dims = Config.P0_HEAD_DIMS if not requires_i else Config.PI_HEAD_DIMS
        
        self.network = MLP(
            input_dim=actual_input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation=output_activation
        )
    
    def forward(self, h: torch.Tensor, i: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            h: (batch, input_dim) - 共享表征
            i: (batch, 1) - 投资成本（仅当 requires_i=True 时使用）
        Returns:
            P: (batch, 1) - 股票价值
        """
        if self.requires_i:
            if i is None:
                raise ValueError("Investment cost i is required for PI head")
            h = torch.cat([h, i], dim=-1)
        
        return self.network(h)


class SharedModel(nn.Module):
    """
    Shared Model: 输出 Q, bp⁰, bpᴵ
    
    结构：
    - ShareLayer: base_state → h
    - Q head: h → Q（不依赖 i）
    - bp⁰ head: h → bp⁰（不依赖 i）
    - bpᴵ head: [h, i] → bpᴵ（依赖 i）
    """
    
    def __init__(
        self,
        base_state_dim: int = 6,
        share_hidden_dims: List[int] = None,
        share_output_dim: int = 64,
        q_hidden_dims: List[int] = None,
        bp_hidden_dims: List[int] = None,
        dropout: float = 0.0,
        share_layer: Optional[ShareLayer] = None
    ):
        super().__init__()
        
        # 共享层
        if share_layer is None:
            self.share_layer = ShareLayer(
                input_dim=base_state_dim,
                hidden_dims=share_hidden_dims,
                output_dim=share_output_dim,
                dropout=dropout
            )
            actual_share_output_dim = share_output_dim
        else:
            self.share_layer = share_layer
            actual_share_output_dim = share_layer.output_dim
        
        # Q head
        self.q_head = QHead(
            input_dim=actual_share_output_dim,
            hidden_dims=q_hidden_dims
        )
        
        # bp⁰ head（不依赖 i）
        self.bp0_head = BpHead(
            input_dim=actual_share_output_dim,
            hidden_dims=bp_hidden_dims,
            requires_i=False
        )
        
        # bpᴵ head（依赖 i）
        self.bpI_head = BpHead(
            input_dim=actual_share_output_dim,
            hidden_dims=bp_hidden_dims,
            requires_i=True
        )
    
    def forward(
        self, 
        firm_state: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            firm_state: (batch, 7) - 完整 firm state
                (b, z, η, i, x, ĉf, ln Kf)
        Returns:
            Q: (batch, 1) - 债券价值
            bp0: (batch, 1) - 不投资时的杠杆候选
            bpI: (batch, 1) - 投资时的杠杆候选
        """
        # 提取 base state（不含 i）
        base_state = torch.cat([
            firm_state[:, :SIMMODEL.I],  # b, z, η
            firm_state[:, SIMMODEL.X:]   # x, ĉf, ln Kf
        ], dim=-1)
        
        # 提取投资成本 i
        i = firm_state[:, SIMMODEL.I:SIMMODEL.I+1]
        
        # 共享表征
        h = self.share_layer(base_state)
        
        # 各 head 输出
        Q = self.q_head(h)
        bp0 = self.bp0_head(h)
        bpI = self.bpI_head(h, i)
        
        return Q, bp0, bpI
    
    def get_Q(self, firm_state: torch.Tensor) -> torch.Tensor:
        """只获取 Q"""
        base_state = torch.cat([
            firm_state[:, :SIMMODEL.I],
            firm_state[:, SIMMODEL.X:]
        ], dim=-1)
        h = self.share_layer(base_state)
        return self.q_head(h)


class CombinedModel(nn.Module):
    """
    Combined Model: 输出 P⁰, Pᴵ, bar_i
    
    结构：
    - ShareLayer: base_state → h
    - P⁰ head: h → P⁰（不依赖 i）
    - Pᴵ head: [h, i] → Pᴵ（依赖 i）
    - bar_i: 由 P⁰, Pᴵ 计算得到
    """
    
    def __init__(
        self,
        base_state_dim: int = 6,
        share_hidden_dims: List[int] = None,
        share_output_dim: int = 64,
        p_hidden_dims: List[int] = None,
        dropout: float = 0.0,
        share_layer: Optional[ShareLayer] = None
    ):
        super().__init__()
        
        # 共享层
        if share_layer is None:
            self.share_layer = ShareLayer(
                input_dim=base_state_dim,
                hidden_dims=share_hidden_dims,
                output_dim=share_output_dim,
                dropout=dropout
            )
            actual_share_output_dim = share_output_dim
        else:
            self.share_layer = share_layer
            actual_share_output_dim = share_layer.output_dim
        
        # P⁰ head（不依赖 i）
        self.p0_head = PHead(
            input_dim=actual_share_output_dim,
            hidden_dims=p_hidden_dims,
            requires_i=False
        )
        
        # Pᴵ head（依赖 i）
        self.pI_head = PHead(
            input_dim=actual_share_output_dim,
            hidden_dims=p_hidden_dims,
            requires_i=True
        )
    
    def forward(
        self, 
        firm_state: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            firm_state: (batch, 7) - 完整 firm state
        Returns:
            P0: (batch, 1) - 不投资时的股票价值
            PI: (batch, 1) - 投资时的股票价值
            bar_i: (batch, 1) - 投资门槛（PI > P0 时为 1）
        """
        # 提取 base state
        base_state = torch.cat([
            firm_state[:, :SIMMODEL.I],
            firm_state[:, SIMMODEL.X:]
        ], dim=-1)
        
        # 提取投资成本 i
        i = firm_state[:, SIMMODEL.I:SIMMODEL.I+1]
        
        # 共享表征
        h = self.share_layer(base_state)
        
        # 各 head 输出
        P0 = self.p0_head(h)
        PI = self.pI_head(h, i)
        
        # 计算投资门槛（软化版本，用于可导训练）
        bar_i = torch.sigmoid(10 * (PI - P0))  # 平滑的指示函数
        
        return P0, PI, bar_i
    
    def get_hard_bar_i(self, firm_state: torch.Tensor) -> torch.Tensor:
        """获取硬投资门槛（用于模拟）"""
        P0, PI, _ = self.forward(firm_state)
        return (PI > P0).float()


class BarzModel(nn.Module):
    """
    Bar_z 模型：计算破产/退出阈值
    
    输入：base_state（不含 i）
    输出：bar_z ∈ [0, 1]
    """
    
    def __init__(
        self,
        input_dim: int = 6,
        hidden_dims: List[int] = None,
        dropout: float = 0.0
    ):
        super().__init__()
        
        if hidden_dims is None:
            hidden_dims = [64, 32, 16]
        
        self.network = MLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation='sigmoid',
            dropout=dropout
        )
    
    def forward(self, base_state: torch.Tensor) -> torch.Tensor:
        """
        Args:
            base_state: (batch, 6) - (b, z, η, x, ĉ, ln K)
        Returns:
            bar_z: (batch, 1) - 破产/退出概率
        """
        return self.network(base_state)


class BariModel(nn.Module):
    """
    Bar_i 模型：价值版本的投资门槛
    用于诊断和一致性检查
    
    输入：base_state（不含 i）
    输出：bar_i_value
    """
    
    def __init__(
        self,
        input_dim: int = 6,
        hidden_dims: List[int] = None,
        dropout: float = 0.0
    ):
        super().__init__()
        
        if hidden_dims is None:
            hidden_dims = [64, 32, 16]
        
        self.network = MLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation='sigmoid',
            dropout=dropout
        )
    
    def forward(self, base_state: torch.Tensor) -> torch.Tensor:
        """
        Args:
            base_state: (batch, 6)
        Returns:
            bar_i_value: (batch, 1)
        """
        return self.network(base_state)
# --- End embedded: models/share_layer.py ---


In [9]:
# --- Begin embedded: models/policy_value.py ---
"""
Policy & Value 统一接口

整合 SharedModel, CombinedModel, BarzModel, BariModel
提供统一的 forward 和工具函数
"""

import torch
import torch.nn as nn
from typing import Dict, Tuple, Optional, NamedTuple



class PolicyValueOutput(NamedTuple):
    """Policy & Value 模型的输出"""
    Q: torch.Tensor          # 债券价值
    bp0: torch.Tensor        # 不投资时杠杆候选
    bpI: torch.Tensor        # 投资时杠杆候选
    P0: torch.Tensor         # 不投资时股票价值
    PI: torch.Tensor         # 投资时股票价值
    bar_i: torch.Tensor      # 投资门槛
    bar_z: torch.Tensor      # 破产门槛
    P: torch.Tensor          # 综合股票价值
    Phat: torch.Tensor       # P hat (中间值)
    bp: torch.Tensor         # 综合杠杆候选


class PolicyValueModel(nn.Module):
    """
    Policy & Value 统一模型
    
    整合：
    - SharedModel: Q, bp0, bpI
    - CombinedModel: P0, PI, bar_i
    - BarzModel: bar_z
    - BariModel: bar_i (value version)
    
    并提供计算 P, Phat, bp 的工具函数
    """
    
    def __init__(
        self,
        base_state_dim: int = 6,
        share_hidden_dims: Optional[list] = None,
        share_output_dim: int = 64,
        dropout: float = 0.0
    ):
        super().__init__()
        
        share_layer = ShareLayer(
            input_dim=base_state_dim,
            hidden_dims=share_hidden_dims,
            output_dim=share_output_dim,
            dropout=dropout
        )
        
        # 主要模型
        self.shared_model = SharedModel(
            base_state_dim=base_state_dim,
            share_hidden_dims=share_hidden_dims,
            share_output_dim=share_output_dim,
            dropout=dropout,
            share_layer=share_layer
        )
        
        self.combined_model = CombinedModel(
            base_state_dim=base_state_dim,
            share_hidden_dims=share_hidden_dims,
            share_output_dim=share_output_dim,
            dropout=dropout,
            share_layer=share_layer
        )
        
        # 辅助模型
        self.barz_model = BarzModel(
            input_dim=base_state_dim,
            dropout=dropout
        )
        
        self.bari_model = BariModel(
            input_dim=base_state_dim,
            dropout=dropout
        )
        
        # 经济参数
        self.register_buffer('delta', torch.tensor(Config.DELTA))
        self.register_buffer('phi', torch.tensor(Config.PHI))
        self.register_buffer('g', torch.tensor(Config.G))
    
    def extract_base_state(self, firm_state: torch.Tensor) -> torch.Tensor:
        """提取 base state（不含 i）"""
        return torch.cat([
            firm_state[:, :SIMMODEL.I],
            firm_state[:, SIMMODEL.X:]
        ], dim=-1)
    
    def forward(
        self, 
        firm_state: torch.Tensor,
        return_all: bool = True
    ) -> PolicyValueOutput:
        """
        完整前向传播
        
        Args:
            firm_state: (batch, 7) - (b, z, η, i, x, ĉf, ln Kf)
            return_all: 是否返回所有输出
        
        Returns:
            PolicyValueOutput: 包含所有输出的命名元组
        """
        # SharedModel 输出
        Q, bp0, bpI = self.shared_model(firm_state)
        
        # CombinedModel 输出
        P0, PI, bar_i = self.combined_model(firm_state)

        # 计算 Phat/P，同时基于积分结果得到 bar_z（P>0 则 bar_z=0，否则=1）
        Phat, P, bar_z = self.cal_phats(firm_state, P0, PI)
        bp = self.cal_bp(bp0, bpI, bar_i)
        
        return PolicyValueOutput(
            Q=Q,
            bp0=bp0,
            bpI=bpI,
            P0=P0,
            PI=PI,
            bar_i=bar_i,
            bar_z=bar_z,
            P=P,
            Phat=Phat,
            bp=bp
        )
    
    def cal_phats(
        self,
        firm_state: torch.Tensor,
        P0: torch.Tensor,
        PI: torch.Tensor,
        simulated_i: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        计算 Phat、P、bar_z

        按 Cal_Phats 思路：保持其他状态不变，枚举 i 的若干取值，重新计算 (P0, PI)，对 max(P0, PI)
        在 i 维度上求平均作为 Phat；P 取其非负部分；bar_z=1(P<=0)，否则 0。
        """
        device = firm_state.device
        batch_size = firm_state.size(0)

        if simulated_i is None:
            # 使用少量均匀点近似积分
            simulated_i = torch.linspace(0.0, Config.I_THRESHOLD, steps=5, device=device).unsqueeze(-1)

        P0_list = []
        PI_list = []
        for i_val in simulated_i:
            modified = firm_state.clone()
            modified[:, SIMMODEL.I] = i_val.expand(batch_size)
            P0_i, PI_i, _ = self.combined_model(modified)
            P0_list.append(P0_i)
            PI_list.append(PI_i)

        P0_stack = torch.stack(P0_list, dim=0)  # (n_i, batch, 1)
        PI_stack = torch.stack(PI_list, dim=0)  # (n_i, batch, 1)
        max_vals = torch.max(P0_stack, PI_stack)
        Phat = max_vals.mean(dim=0)  # (batch, 1)

        P = torch.clamp(Phat, min=0.0)
        # 使用平滑指示避免梯度中断：bar_z ≈ 1(P<=0) via sigmoid with large slope
        bar_z = torch.sigmoid(-50.0 * P)

        return Phat, P, bar_z
    
    def cal_bp(
        self,
        bp0: torch.Tensor,
        bpI: torch.Tensor,
        bar_i: torch.Tensor
    ) -> torch.Tensor:
        """
        计算综合杠杆候选
        
        bp = bar_i * bpI + (1 - bar_i) * bp0
        """
        return bar_i * bpI + (1 - bar_i) * bp0
    
    def update_leverage(
        self,
        b_old: torch.Tensor,
        bp: torch.Tensor,
        eta: torch.Tensor
    ) -> torch.Tensor:
        """
        更新杠杆
        
        b_new = η * bp + (1 - η) * b_old
        
        Args:
            b_old: 旧杠杆
            bp: 杠杆候选
            eta: 再融资开关
        
        Returns:
            b_new: 新杠杆
        """
        return eta * bp + (1 - eta) * b_old
    
    def update_capital(
        self,
        K_old: torch.Tensor,
        bar_i: torch.Tensor
    ) -> torch.Tensor:
        """
        更新资本
        
        K_new = bar_i * G * K_old + (1 - bar_i) * K_old
        
        Args:
            K_old: 旧资本
            bar_i: 投资决策
        
        Returns:
            K_new: 新资本
        """
        return bar_i * self.g * K_old + (1 - bar_i) * K_old
    
    def get_shared_output(
        self, 
        firm_state: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """只获取 SharedModel 输出"""
        return self.shared_model(firm_state)
    
    def get_combined_output(
        self, 
        firm_state: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """只获取 CombinedModel 输出"""
        return self.combined_model(firm_state)
    
    def get_bar_z(self, firm_state: torch.Tensor) -> torch.Tensor:
        """只获取 bar_z"""
        base_state = self.extract_base_state(firm_state)
        return self.barz_model(base_state)
    
    def get_bar_i_value(self, firm_state: torch.Tensor) -> torch.Tensor:
        """获取 bar_i 的 value version"""
        base_state = self.extract_base_state(firm_state)
        return self.bari_model(base_state)
    
    def freeze(self):
        """冻结所有参数"""
        for param in self.parameters():
            param.requires_grad = False
        self.eval()
    
    def unfreeze(self):
        """解冻所有参数"""
        for param in self.parameters():
            param.requires_grad = True
        self.train()
    
    def get_module_parameters(self, module_name: str):
        """
        获取指定模块的参数
        
        Args:
            module_name: 'shared', 'combined', 'barz', 'bari'
        """
        modules = {
            'shared': self.shared_model,
            'combined': self.combined_model,
            'barz': self.barz_model,
            'bari': self.bari_model
        }
        
        if module_name not in modules:
            raise ValueError(f"Unknown module: {module_name}")
        
        return modules[module_name].parameters()


class CalPhats:
    """
    计算 Phat, P, bar_z 的工具类
    
    用于在训练和模拟中统一计算逻辑
    """
    
    def __init__(self, combined_model: CombinedModel, barz_model: BarzModel):
        self.combined_model = combined_model
        self.barz_model = barz_model
    
    def __call__(
        self, 
        firm_state: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        计算 bar_z, Phat, P
        
        Args:
            firm_state: (batch, 7)
        
        Returns:
            bar_z: (batch, 1)
            Phat: (batch, 1)
            P: (batch, 1)
        """
        # Combined model 输出
        P0, PI, bar_i = self.combined_model(firm_state)
        
        # bar_z
        base_state = torch.cat([
            firm_state[:, :SIMMODEL.I],
            firm_state[:, SIMMODEL.X:]
        ], dim=-1)
        bar_z = self.barz_model(base_state)
        
        # Phat 和 P
        Phat = bar_i * PI + (1 - bar_i) * P0
        P = (1 - bar_z) * Phat
        
        return bar_z, Phat, P
# --- End embedded: models/policy_value.py ---


In [10]:
# --- Begin embedded: models/sdf_fc1.py ---
"""
SDF & FC1 模型

SDF: 随机贴现因子网络
FC1: 宏观状态预测网络（FC1_C 预测 ĉf，FC1_K 预测 ln Kf）

关键口径：
- FC1 输出默认在标准化空间
- 写回 df 前必须 inverse_transform
- df 中保存的一律是 physical scale

树状分支结构（支持任意 N 个分支）：
- parent: (w_t, k_t, c_t) → t 期父节点
- children: [(w_{t+1}^{(j)}, k_{t+1}^{(j)}, c_{t+1}^{(j)})] → N 条模拟路径
- 每条路径中 η 通过伯努利分布随机抽取

SDF 计算（对每条路径 j）：
    M^{(j)} = β^κ * exp((k_{t+1}^{(j)} - k_t)*(-γ) - κ/σ*(c_{t+1}^{(j)} - c_t)) 
              * (w_{t+1}^{(j)} / (w_t - exp(c_t)))^(κ-1)
"""

import torch
import torch.nn as nn
from typing import Tuple, Optional, List, Union



def compute_sdf(
    w_parent: torch.Tensor,
    w_children: Union[List[torch.Tensor], torch.Tensor],
    k_parent: torch.Tensor,
    k_children: Union[List[torch.Tensor], torch.Tensor],
    c_parent: torch.Tensor,
    c_children: Union[List[torch.Tensor], torch.Tensor],
    beta: float = None,
    gamma: float = None,
    kappa: float = None,
    sigma: float = None,
    eps: float = 1e-3
) -> Union[List[torch.Tensor], torch.Tensor]:
    """
    计算 SDF（支持任意数量的分支路径）
    
    M^{(j)} = β^κ * exp((k_{t+1}^{(j)} - k_t)*(-γ) - κ/σ*(c_{t+1}^{(j)} - c_t)) 
              * (w_{t+1}^{(j)} / (w_t - exp(c_t)))^(κ-1)
    
    Args:
        w_parent: (batch,) 父节点价值函数 w_t
        w_children: List[(batch,)] 或 (batch, n_branches) 或 (batch, n_branches, 1)
        k_parent: (batch,) 父节点资本对数 ln K_t
        k_children: List[(batch,)] 或 (batch, n_branches) 或 (batch, n_branches, 1)
        c_parent: (batch,) 父节点消费对数 ĉ_t
        c_children: List[(batch,)] 或 (batch, n_branches) 或 (batch, n_branches, 1)
        beta, gamma, kappa, sigma: 模型参数（默认从 Config 获取）
        eps: 数值稳定项
    
    Returns:
        M_list 或 M_tensor: List[torch.Tensor] 或 (batch, n_branches) 张量
    """
    # 从配置获取默认参数
    if beta is None:
        beta = Config.BETA
    if gamma is None:
        gamma = Config.GAMMA
    if kappa is None:
        kappa = Config.KAPPA
    if sigma is None:
        sigma = Config.SIGMA

    # β^κ
    tmp = beta ** kappa

    # 张量路径（批量分支）
    if isinstance(w_children, torch.Tensor):
        w_children_t = w_children
        k_children_t = k_children
        c_children_t = c_children

        if w_children_t.dim() == 3 and w_children_t.size(-1) == 1:
            w_children_t = w_children_t.squeeze(-1)
            k_children_t = k_children_t.squeeze(-1)
            c_children_t = c_children_t.squeeze(-1)
        if w_children_t.dim() == 1:
            w_children_t = w_children_t.unsqueeze(-1)
            k_children_t = k_children_t.unsqueeze(-1)
            c_children_t = c_children_t.unsqueeze(-1)

        w_parent_t = w_parent.squeeze(-1) if w_parent.dim() > 1 else w_parent
        k_parent_t = k_parent.squeeze(-1) if k_parent.dim() > 1 else k_parent
        c_parent_t = c_parent.squeeze(-1) if c_parent.dim() > 1 else c_parent

        denom = (w_parent_t - torch.exp(c_parent_t)).clamp_min(eps).unsqueeze(-1)
        ratio = (w_children_t / denom).clamp_min(eps)

        M = torch.exp(
            (k_children_t - k_parent_t.unsqueeze(-1)) * (-gamma)
            - kappa / sigma * (c_children_t - c_parent_t.unsqueeze(-1))
        ) * torch.pow(ratio, kappa - 1) * tmp

        return M

    # List 路径（逐分支）
    # 当期"财富-消费"
    denom = (w_parent - torch.exp(c_parent)).clamp_min(eps)

    # 对每条路径计算 M
    M_list = []
    for w_child, k_child, c_child in zip(w_children, k_children, c_children):
        ratio = (w_child / denom).clamp_min(eps)
        M = torch.exp(
            (k_child - k_parent) * (-gamma) - kappa / sigma * (c_child - c_parent)
        ) * torch.pow(ratio, kappa - 1) * tmp
        M_list.append(M)

    return M_list


def compute_sdf_legacy(
    w: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    kf: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    cf: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    beta: float = None,
    gamma: float = None,
    kappa: float = None,
    sigma: float = None,
    eps: float = 1e-3
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    计算 SDF（旧版接口，兼容2分支情况）
    
    Args:
        w: (w1, w2, w3) = (w_parent, w_child1, w_child2)
        kf: (k1, k2, k3) = (k_parent, k_child1, k_child2)
        cf: (c1, c2, c3) = (c_parent, c_child1, c_child2)
    
    Returns:
        M1, M2: 两条路径的 SDF
    """
    w1, w2, w3 = w
    k1, k2, k3 = kf
    c1, c2, c3 = cf
    
    M_list = compute_sdf(
        w_parent=w1,
        w_children=[w2, w3],
        k_parent=k1,
        k_children=[k2, k3],
        c_parent=c1,
        c_children=[c2, c3],
        beta=beta,
        gamma=gamma,
        kappa=kappa,
        sigma=sigma,
        eps=eps
    )
    
    return M_list[0], M_list[1]


class SDFModel(nn.Module):
    """
    随机贴现因子 (SDF) 网络
    
    用于学习 M_{t,t+1}，满足 Euler 方程约束
    E_t[M_{t,t+1} R_{t+1}] = 1
    
    支持任意数量的分支路径：
        M^{(j)} = β^κ * exp((k_{t+1}^{(j)} - k_t)*(-γ) - κ/σ*(c_{t+1}^{(j)} - c_t)) 
                  * (w_{t+1}^{(j)} / (w_t - exp(c_t)))^(κ-1)
    """
    
    def __init__(
        self,
        input_dim: int = 4,  # (c_t, c_{t+1}, k_t, k_{t+1}) 或其它配置
        hidden_dims: List[int] = None,
        output_positive: bool = True
    ):
        super().__init__()
        
        if hidden_dims is None:
            hidden_dims = Config.SDF_HIDDEN_DIMS
        
        # SDF 主网络
        self.network = MLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation='softplus' if output_positive else None
        )
        
        # 用于计算 M 的参数
        self.beta = Config.BETA
        self.gamma = Config.GAMMA
        self.kappa = Config.KAPPA
        self.sigma = Config.SIGMA
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        计算 SDF
        
        Args:
            x: 输入特征
        Returns:
            M: SDF 值
        """
        return self.network(x)
    
    def get_M(
        self,
        w_parent: torch.Tensor,
        w_children: Union[List[torch.Tensor], torch.Tensor],
        k_parent: torch.Tensor,
        k_children: Union[List[torch.Tensor], torch.Tensor],
        c_parent: torch.Tensor,
        c_children: Union[List[torch.Tensor], torch.Tensor],
        eps: float = 1e-3
    ) -> List[torch.Tensor]:
        """
        根据价值函数和状态变量计算 SDF（支持任意分支数）
        
        Args:
            w_parent: (batch,) 父节点价值函数
            w_children: List[(batch,)] 或 (batch, n_branches) 子节点价值函数
            k_parent: (batch,) 父节点资本对数
            k_children: 子节点资本对数
            c_parent: (batch,) 父节点消费对数
            c_children: 子节点消费对数
            eps: 数值稳定项
        
        Returns:
            M_list: List[torch.Tensor] - 每条路径的 SDF
        """
        return compute_sdf(
            w_parent=w_parent,
            w_children=w_children,
            k_parent=k_parent,
            k_children=k_children,
            c_parent=c_parent,
            c_children=c_children,
            beta=self.beta,
            gamma=self.gamma,
            kappa=self.kappa,
            sigma=self.sigma,
            eps=eps
        )
    
    def get_M_legacy(
        self,
        w: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        kf: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        cf: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        eps: float = 1e-3
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        旧版接口，兼容2分支情况
        
        Args:
            w: (w1, w2, w3) = (w_parent, w_child1, w_child2)
            kf: (k1, k2, k3)
            cf: (c1, c2, c3)
        
        Returns:
            M1, M2
        """
        return compute_sdf_legacy(
            w, kf, cf,
            beta=self.beta,
            gamma=self.gamma,
            kappa=self.kappa,
            sigma=self.sigma,
            eps=eps
        )
    
    def compute_sdf_from_consumption(
        self, 
        c_t: torch.Tensor, 
        c_t1: torch.Tensor,
        k_t: torch.Tensor,
        k_t1: torch.Tensor
    ) -> torch.Tensor:
        """
        根据消费增长计算简单 SDF (CRRA 形式)
        
        标准 CRRA 形式：M = β * (C_{t+1}/C_t)^{-γ}
        """
        # 对数消费增长
        log_c_growth = torch.log(c_t1 + 1e-8) - torch.log(c_t + 1e-8)
        
        # CRRA SDF
        M = self.beta * torch.exp(-self.gamma * log_c_growth)
        
        return M


class FC1Model(nn.Module):
    """
    FC1 模型：宏观状态预测
    
    包含两个子网络：
    - FC1_C: 预测 ĉf（消费相关宏观状态）
    - FC1_K: 预测 ln Kf（资本对数）
    
    输入维度: 4 = (x_{t-1}, x_t, ĉf_{t-1}, ln Kf_{t-1})
    """
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dims: List[int] = None
    ):
        super().__init__()
        
        if input_dim is None:
            input_dim = Config.FC1_INPUT_DIM
        if hidden_dims is None:
            hidden_dims = Config.FC1_HIDDEN_DIMS
        
        # FC1_C: 预测 ĉf
        self.FC1_C = MLPWithScaler(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1
        )
        
        # FC1_K: 预测 ln Kf
        self.FC1_K = MLPWithScaler(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1
        )
    
    def fit_scalers(
        self, 
        x: torch.Tensor, 
        hatcf: torch.Tensor, 
        lnkf: torch.Tensor
    ):
        """
        拟合两个子网络的标准化参数
        
        Args:
            x: 输入特征
            hatcf: ĉf 目标值
            lnkf: ln Kf 目标值
        """
        self.FC1_C.fit_scaler(x, hatcf)
        self.FC1_K.fit_scaler(x, lnkf)
    
    def forward(
        self, 
        x: torch.Tensor,
        return_physical: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        前向传播
        
        Args:
            x: (batch, 4) - (x_{t-1}, x_t, ĉf_{t-1}, ln Kf_{t-1})
            return_physical: 是否返回物理尺度（默认 True）
        
        Returns:
            hatcf: (batch, 1) - 预测的 ĉf_{t+1}
            lnkf: (batch, 1) - 预测的 ln Kf_{t+1}
        """
        if return_physical:
            hatcf = self.FC1_C.forward_physical(x)
            lnkf = self.FC1_K.forward_physical(x)
        else:
            hatcf = self.FC1_C.forward_normalized(x)
            lnkf = self.FC1_K.forward_normalized(x)
        
        return hatcf, lnkf
    
    def forward_normalized(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """返回标准化空间的输出"""
        return self.forward(x, return_physical=False)
    
    def forward_physical(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """返回物理尺度的输出"""
        return self.forward(x, return_physical=True)


class SDFFC1Combined(nn.Module):
    """
    SDF & FC1 组合模型
    
    统一管理 SDF 和 FC1 的训练与推理
    """
    # comment: 这个模型的核心输入是 (x_{t-1}, x_t, ĉf_{t-1}, lnKf_{t-1})，
    #          先用 FC1 输入为 (x_{t-1}, x_t, ĉf_{t-1}, lnKf_{t-1}) 预测 ĉf_t, lnKf_t，
    #          再用 ValueFunctionW 计算 w_{t-1}, w_t，最后用 w_{t-1}, w_t 计算 M(t-1, t)。
    def __init__(
        self,
        sdf_input_dim: int = 4,
        fc1_input_dim: int = 4,
        sdf_hidden_dims: List[int] = None,
        fc1_hidden_dims: List[int] = None,
        w_hidden_dims: List[int] = None
    ):
        super().__init__()
        
        self.sdf_model = SDFModel(
            input_dim=sdf_input_dim,
            hidden_dims=sdf_hidden_dims
        )
        
        self.fc1_model = FC1Model(
            input_dim=fc1_input_dim,
            hidden_dims=fc1_hidden_dims
        )
        
        self.value_model = ValueFunctionW(
            input_dim=3,
            hidden_dims=w_hidden_dims
        )
    
    @property
    def FC1_C(self):
        return self.fc1_model.FC1_C
    
    @property
    def FC1_K(self):
        return self.fc1_model.FC1_K
    
    def get_M_legacy(
        self,
        w: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        kf: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        cf: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        eps: float = 1e-3
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """旧版接口，兼容2分支情况"""
        return self.sdf_model.get_M_legacy(w, kf, cf, eps)
    
    def forward_fc1(
        self, 
        x: torch.Tensor,
        return_physical: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """预测宏观状态（输入: x_{t-1}, x_t, ĉf_{t-1}, lnKf_{t-1}）"""
        return self.fc1_model(x, return_physical)
    
    def fit_fc1_scalers(
        self, 
        x: torch.Tensor, 
        hatcf: torch.Tensor, 
        lnkf: torch.Tensor
    ):
        """拟合 FC1 标准化参数"""
        self.fc1_model.fit_scalers(x, hatcf, lnkf)

    def forward_step(
        self,
        x_prev: torch.Tensor,
        x_curr: torch.Tensor,
        hatcf_prev: torch.Tensor,
        lnkf_prev: torch.Tensor,
        return_physical: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        使用 (x_{t-1}, x_t, ĉf_{t-1}, lnKf_{t-1}) 预测 ĉf_t、lnKf_t 并计算 M(t-1,t)。
        
        Returns:
            w_prev, w_curr, M, hatcf_curr, lnkf_curr
        """
        def to_col(t: torch.Tensor) -> torch.Tensor:
            return t if t.dim() > 1 else t.unsqueeze(-1)
        
        x_prev = to_col(x_prev)
        x_curr = to_col(x_curr)
        hatcf_prev = to_col(hatcf_prev)
        lnkf_prev = to_col(lnkf_prev)

        # 多分支批量路径：(batch, n_children, 1)
        if x_curr.dim() == 3:
            batch, n_children, _ = x_curr.shape

            x_prev_base = x_prev[:, 0, :] if x_prev.dim() == 3 else x_prev
            hatcf_prev_base = hatcf_prev[:, 0, :] if hatcf_prev.dim() == 3 else hatcf_prev
            lnkf_prev_base = lnkf_prev[:, 0, :] if lnkf_prev.dim() == 3 else lnkf_prev

            if x_prev.dim() == 2:
                x_prev = x_prev.unsqueeze(1).expand(-1, n_children, -1)
            if hatcf_prev.dim() == 2:
                hatcf_prev = hatcf_prev.unsqueeze(1).expand(-1, n_children, -1)
            if lnkf_prev.dim() == 2:
                lnkf_prev = lnkf_prev.unsqueeze(1).expand(-1, n_children, -1)

            fc1_input = torch.cat([x_prev, x_curr, hatcf_prev, lnkf_prev], dim=-1)
            flat = fc1_input.reshape(-1, fc1_input.shape[-1])
            hatcf_curr, lnkf_curr = self.forward_fc1(flat, return_physical=return_physical)
            hatcf_curr = hatcf_curr.view(batch, n_children, 1)
            lnkf_curr = lnkf_curr.view(batch, n_children, 1)

            w_prev = self.value_model(torch.cat([x_prev_base, hatcf_prev_base, lnkf_prev_base], dim=-1))
            w_curr_input = torch.cat([x_curr, hatcf_curr, lnkf_curr], dim=-1)
            w_curr = self.value_model(w_curr_input.reshape(-1, w_curr_input.shape[-1]))
            w_curr = w_curr.view(batch, n_children, 1)

            M = self.sdf_model.get_M(
                w_parent=w_prev.squeeze(-1),
                w_children=w_curr.squeeze(-1),
                k_parent=lnkf_prev_base.squeeze(-1),
                k_children=lnkf_curr.squeeze(-1),
                c_parent=hatcf_prev_base.squeeze(-1),
                c_children=hatcf_curr.squeeze(-1)
            )
            if isinstance(M, list):
                M = M[0]
            if M.dim() == 2:
                M = M.unsqueeze(-1)

            return w_prev, w_curr, M, hatcf_curr, lnkf_curr

        # 单分支路径：(batch, 1)
        fc1_input = torch.cat([x_prev, x_curr, hatcf_prev, lnkf_prev], dim=-1)
        hatcf_curr, lnkf_curr = self.forward_fc1(fc1_input, return_physical=return_physical)

        w_prev = self.value_model(torch.cat([x_prev, hatcf_prev, lnkf_prev], dim=-1))
        w_curr = self.value_model(torch.cat([x_curr, hatcf_curr, lnkf_curr], dim=-1))

        M_list = self.sdf_model.get_M(
            w_parent=w_prev,
            w_children=[w_curr],
            k_parent=lnkf_prev,
            k_children=[lnkf_curr],
            c_parent=hatcf_prev,
            c_children=[hatcf_curr]
        )

        return w_prev, w_curr, M_list[0], hatcf_curr, lnkf_curr


class ValueFunctionW(nn.Module):
    """
    价值函数 W 网络
    
    用于 SDF Loss 中的价值函数部分
    输出 w = (w_t, w_{t+1}^0, w_{t+1}^1, ...)
    """
    
    def __init__(
        self,
        input_dim: int = 2,  # (c, k) 或类似
        hidden_dims: List[int] = None,
        output_positive: bool = True
    ):
        super().__init__()
        
        if hidden_dims is None:
            hidden_dims = [64, 32]
        
        self.network = MLP(
            input_dim=input_dim,
            hidden_dims=hidden_dims,
            output_dim=1,
            output_activation='softplus' if output_positive else None
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, input_dim)
        Returns:
            w: (batch, 1) - 价值函数值
        """
        return self.network(x)
# --- End embedded: models/sdf_fc1.py ---


In [11]:
# --- Begin embedded: losses/sdf_loss.py ---
"""
SDF Loss: 随机贴现因子损失

核心目标：拟合欧拉方程约束，确保所有路径下 SDF 残差同时为零，
叠加 M 的矩约束惩罚，保证统计性质合理。

树状分支结构（支持任意 N 个分支）：
- parent: (w_t, k_t, c_t) → t 期父节点
- children: [(w_{t+1}^{(j)}, k_{t+1}^{(j)}, c_{t+1}^{(j)})] → N 条模拟路径
- 每条路径中 η 通过伯努利分布随机抽取

SDF 计算（对每条路径 j）：
    M^{(j)} = β^κ * exp((k_{t+1}^{(j)} - k_t)*(-γ) - κ/σ*(c_{t+1}^{(j)} - c_t)) 
              * (w_{t+1}^{(j)} / (w_t - exp(c_t)))^(κ-1)

欧拉方程残差（对每条路径 j）：
    loss^{(j)} = exp_term^{(j)} * w_{t+1}^{(j)κ} - (w_t - exp(c_t))^κ = 0

主损失：所有路径残差的乘积（要求同时为零）
L_SDF = main_loss + Σ_j (L1_{M^{(j)}} + L2_{M^{(j)}})
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Dict, Optional, List, Union




def moment_penalty(
    M: torch.Tensor,
    mu_lo: float,
    mu_hi: float,
    var_hi: float,
    delta: float = 1e-3,
    eps: float = 1e-6,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    计算随机贴现因子 M 的矩约束惩罚项。

    对 M 的对数均值和对数方差施加约束：
    - 一阶矩（均值）：mu_lo <= log(E[M]) <= mu_hi
    - 二阶矩（方差）：log(Var[M]) <= var_hi

    Args:
        M: 随机贴现因子张量，shape [batch]
        mu_lo: 对数均值的下界
        mu_hi: 对数均值的上界
        var_hi: 对数方差的上界
        delta: 平滑阶跃函数的平滑度参数
        eps: 数值稳定项，避免 log(0)

    Returns:
        L1: 一阶矩惩罚项（均值约束）
        L2: 二阶矩惩罚项（方差约束）
    """
    # 计算均值和方差
    mu = M.mean()  # E[M]
    var = (M * M).mean() - mu * mu  # Var[M] = E[M^2] - E[M]^2

    # 平滑阶跃函数
    step = lambda x: softplus_gate(x, delta)

    # 一阶矩惩罚：当 log(E[M]) 超出 [mu_lo, mu_hi] 时施加惩罚
    log_mu = torch.log(mu + eps)
    L_low = step(mu_lo - log_mu) * 100.0 * (log_mu - mu_lo) ** 2
    L_high = step(log_mu - mu_hi) * 100.0 * (log_mu - mu_hi) ** 2
    L1 = L_low + L_high

    # 二阶矩惩罚：当 log(Var[M]) > var_hi 时施加惩罚
    log_var = torch.log(var + eps)
    L2 = step(log_var - var_hi) * 100.0 * (log_var - var_hi) ** 2

    return L1, L2


class SDFLoss(nn.Module):
    """
    SDF 损失函数（支持任意分支数）
    
    基于欧拉方程残差和矩约束
    """
    
    def __init__(
        self,
        gamma: float = None,
        kappa: float = None,
        sigma: float = None,
        beta: float = None,
        mu_lo: float = -0.025,
        mu_hi: float = 0.0,
        var_hi: float = 0.25
    ):
        super().__init__()
        
        # 从配置获取参数
        self.gamma = gamma if gamma is not None else Config.GAMMA
        self.kappa = kappa if kappa is not None else Config.KAPPA
        self.sigma = sigma if sigma is not None else Config.SIGMA
        self.beta = beta if beta is not None else Config.BETA
        self.mu_lo = mu_lo
        self.mu_hi = mu_hi
        self.var_hi = var_hi
        
        # 预计算 β^κ
        self.tmp = self.beta ** self.kappa
    
    def compute_euler_residuals(
        self,
        w_parent: torch.Tensor,
        w_children: Union[List[torch.Tensor], torch.Tensor],
        k_parent: torch.Tensor,
        k_children: Union[List[torch.Tensor], torch.Tensor],
        c_parent: torch.Tensor,
        c_children: Union[List[torch.Tensor], torch.Tensor],
    ) -> Union[List[torch.Tensor], torch.Tensor]:
        """
        计算所有路径的欧拉方程残差
        
        loss^{(j)} = exp_term^{(j)} * w_{t+1}^{(j)κ} - (w_t - exp(c_t))^κ
        
        Args:
            w_parent: (batch,) 父节点价值函数
            w_children: List[(batch,)] 子节点价值函数列表
            k_parent: (batch,) 父节点资本对数
            k_children: List[(batch,)] 子节点资本对数列表
            c_parent: (batch,) 父节点消费对数
            c_children: List[(batch,)] 子节点消费对数列表
        
        Returns:
            residuals: List[torch.Tensor] 或 (batch, n_children) 张量
        """
        # 张量路径：(batch, n_children) 或 (batch, n_children, 1)
        if isinstance(w_children, torch.Tensor):
            w_children_t = w_children
            k_children_t = k_children
            c_children_t = c_children

            if w_children_t.dim() == 3 and w_children_t.size(-1) == 1:
                w_children_t = w_children_t.squeeze(-1)
                k_children_t = k_children_t.squeeze(-1)
                c_children_t = c_children_t.squeeze(-1)
            if w_children_t.dim() == 1:
                w_children_t = w_children_t.unsqueeze(-1)
                k_children_t = k_children_t.unsqueeze(-1)
                c_children_t = c_children_t.unsqueeze(-1)

            w_parent_t = w_parent.squeeze(-1) if w_parent.dim() > 1 else w_parent
            k_parent_t = k_parent.squeeze(-1) if k_parent.dim() > 1 else k_parent
            c_parent_t = c_parent.squeeze(-1) if c_parent.dim() > 1 else c_parent

            current_value = (w_parent_t - torch.exp(c_parent_t)).clamp_min(1e-8)
            current_value_kappa = torch.pow(current_value, self.kappa).unsqueeze(-1)

            exp_term = torch.exp(
                (k_children_t - k_parent_t.unsqueeze(-1)) * (1.0 - self.gamma)
                - self.kappa / self.sigma * (c_children_t - c_parent_t.unsqueeze(-1))
            ) * self.tmp

            residuals = exp_term * torch.pow(w_children_t, self.kappa) - current_value_kappa
            return residuals

        # List 路径（逐分支）
        current_value = (w_parent - torch.exp(c_parent)).clamp_min(1e-8)
        current_value_kappa = torch.pow(current_value, self.kappa)

        residuals = []
        for w_child, k_child, c_child in zip(w_children, k_children, c_children):
            exp_term = torch.exp(
                (k_child - k_parent) * (1.0 - self.gamma)
                - self.kappa / self.sigma * (c_child - c_parent)
            ) * self.tmp
            residual = exp_term * torch.pow(w_child, self.kappa) - current_value_kappa
            residuals.append(residual)

        return residuals
    
    def forward(
        self,
        w_parent: torch.Tensor,
        w_children: List[torch.Tensor],
        M_list: List[torch.Tensor],
        k_parent: torch.Tensor,
        k_children: List[torch.Tensor],
        c_parent: torch.Tensor,
        c_children: List[torch.Tensor],
    ) -> torch.Tensor:
        """
        计算 SDF 损失（支持任意分支数）
        
        Args:
            w_parent: (batch,) 父节点价值函数
            w_children: List[(batch,)] 子节点价值函数列表
            M_list: List[(batch,)] 每条路径的 SDF
            k_parent: (batch,) 父节点资本对数
            k_children: List[(batch,)] 子节点资本对数列表
            c_parent: (batch,) 父节点消费对数
            c_children: List[(batch,)] 子节点消费对数列表
        
        Returns:
            final_loss: 总损失
        """
        # 1. 计算所有路径的欧拉方程残差
        residuals = self.compute_euler_residuals(
            w_parent, w_children,
            k_parent, k_children,
            c_parent, c_children
        )
        
        # 2. 主损失：所有路径残差的乘积（要求同时为零）
        if isinstance(residuals, list):
            raw_loss = residuals[0]
            for residual in residuals[1:]:
                raw_loss = raw_loss * residual
        else:
            raw_loss = residuals.prod(dim=-1)
        
        # 使用 log1p 平滑
        main_loss = torch.mean(torch.log1p(raw_loss.abs()))
        
        # 3. 矩约束惩罚（所有路径的 M）
        moment_loss = torch.tensor(0.0, device=w_parent.device)
        if isinstance(M_list, torch.Tensor):
            M = M_list
            if M.dim() == 3 and M.size(-1) == 1:
                M = M.squeeze(-1)
            if M.dim() == 1:
                M = M.unsqueeze(-1)
            for j in range(M.shape[1]):
                L1, L2 = moment_penalty(M[:, j], self.mu_lo, self.mu_hi, self.var_hi)
                moment_loss = moment_loss + L1 + L2
        else:
            for M in M_list:
                L1, L2 = moment_penalty(M, self.mu_lo, self.mu_hi, self.var_hi)
                moment_loss = moment_loss + L1 + L2
        
        # 总损失
        final_loss = main_loss + moment_loss
        
        return final_loss
    
    def forward_legacy(
        self,
        outputs: Tuple[
            Tuple[torch.Tensor, torch.Tensor, torch.Tensor],  # w
            torch.Tensor,  # M1
            torch.Tensor,  # M2
            Tuple[torch.Tensor, torch.Tensor, torch.Tensor],  # kf
            Tuple[torch.Tensor, torch.Tensor, torch.Tensor],  # cf
        ],
        c: None = None,
        k: None = None,
    ) -> torch.Tensor:
        """
        旧版接口，兼容2分支情况
        
        Args:
            outputs: (w, M1, M2, kf, cf) 格式
        """
        w, M1, M2, kf, cf = outputs
        w1, w2, w3 = w
        k1, k2, k3 = kf
        c1, c2, c3 = cf
        
        return self.forward(
            w_parent=w1,
            w_children=[w2, w3],
            M_list=[M1, M2],
            k_parent=k1,
            k_children=[k2, k3],
            c_parent=c1,
            c_children=[c2, c3]
        )
    
    def forward_with_details(
        self,
        w_parent: torch.Tensor,
        w_children: List[torch.Tensor],
        M_list: List[torch.Tensor],
        k_parent: torch.Tensor,
        k_children: List[torch.Tensor],
        c_parent: torch.Tensor,
        c_children: List[torch.Tensor],
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        计算 SDF 损失并返回详细信息
        """
        # 计算残差
        residuals = self.compute_euler_residuals(
            w_parent, w_children,
            k_parent, k_children,
            c_parent, c_children
        )
        
        # 主损失
        if isinstance(residuals, list):
            raw_loss = residuals[0]
            for residual in residuals[1:]:
                raw_loss = raw_loss * residual
        else:
            raw_loss = residuals.prod(dim=-1)
        main_loss = torch.mean(torch.log1p(raw_loss.abs()))
        
        # 矩约束惩罚
        loss_dict = {'main_loss': main_loss}
        moment_loss = torch.tensor(0.0, device=w_parent.device)
        
        if isinstance(M_list, torch.Tensor):
            M = M_list
            if M.dim() == 3 and M.size(-1) == 1:
                M = M.squeeze(-1)
            if M.dim() == 1:
                M = M.unsqueeze(-1)
            for j in range(M.shape[1]):
                L1, L2 = moment_penalty(M[:, j], self.mu_lo, self.mu_hi, self.var_hi)
                loss_dict[f'L1_M{j+1}'] = L1
                loss_dict[f'L2_M{j+1}'] = L2
                loss_dict[f'M{j+1}_mean'] = M[:, j].mean()
                moment_loss = moment_loss + L1 + L2
        else:
            for j, M in enumerate(M_list):
                L1, L2 = moment_penalty(M, self.mu_lo, self.mu_hi, self.var_hi)
                loss_dict[f'L1_M{j+1}'] = L1
                loss_dict[f'L2_M{j+1}'] = L2
                loss_dict[f'M{j+1}_mean'] = M.mean()
                moment_loss = moment_loss + L1 + L2
        
        # 残差信息
        if isinstance(residuals, list):
            for j, residual in enumerate(residuals):
                loss_dict[f'residual{j+1}_mean'] = residual.mean()
        else:
            for j in range(residuals.shape[1]):
                loss_dict[f'residual{j+1}_mean'] = residuals[:, j].mean()
        
        final_loss = main_loss + moment_loss
        
        return final_loss, loss_dict


# 便捷函数
def SDFloss(outputs, c=None, k=None):
    """函数式接口（旧版兼容）"""
    loss_fn = SDFLoss()
    return loss_fn.forward_legacy(outputs, c, k)
# --- End embedded: losses/sdf_loss.py ---


In [12]:
# --- Begin embedded: losses/p0_loss.py ---
"""
P0 Loss: 股价相关损失（不投资情形）

核心目标：拟合股价 Bellman 方程残差与债务选择 FOC 残差，
叠加单调性惩罚与 z 值惩罚，确保经济意义一致性。

支持任意数量的分支路径：
- parent: (P0_t, CF0p_t, ...) → t 期父节点
- children: [(P_{t+1}^{(j)}, bar_z_{t+1}^{(j)}, ...)] → N 条模拟路径

L_P0 = loss_bellman + loss_foc + mono_penalty
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Dict, Optional, List




class P0Loss(nn.Module):
    """
    P0（不投资时股价）损失函数
    
    包含：
    - Bellman 残差损失
    - FOC 残差损失（债务选择一阶条件）
    - 单调性惩罚
    - z 值惩罚
    """
    
    def __init__(
        self,
        delta: float = None,
        tau: float = None,
        kappa_b: float = None,
        kappa_e: float = None,
        aio_weight: float = None,
        alpha_z: float = None,
        beta_z: float = None,
        z0: float = None,
        lambda_b: float = 1.0,
        lambda_z: float = 1.0
    ):
        super().__init__()
        
        # 经济参数
        self.delta = delta if delta is not None else Config.DELTA
        self.tau = tau if tau is not None else Config.TAU
        self.kappa_b = kappa_b if kappa_b is not None else Config.KAPPA_B
        self.kappa_e = kappa_e if kappa_e is not None else Config.KAPPA_E
        
        # 损失权重
        self.aio_weight = aio_weight if aio_weight is not None else Config.AIO_WEIGHT
        self.alpha_z = alpha_z if alpha_z is not None else Config.ALPHA_Z
        self.beta_z = beta_z if beta_z is not None else Config.BETA_Z
        self.z0 = z0 if z0 is not None else Config.Z0
        self.lambda_b = lambda_b
        self.lambda_z = lambda_z
    
    def compute_cashflow_p0(
        self,
        x: torch.Tensor,
        z: torch.Tensor,
        b: torch.Tensor,
        Q: torch.Tensor,
        Qp: torch.Tensor,
        eta: torch.Tensor
    ) -> torch.Tensor:
        """
        计算不投资时的股东现金流
        
        CF0p = prod + ((1-κb)Qp - Q) * η + κe * max(-CF0p, 0) * CF0p
        """
        # 税后利润
        prod = compute_cashflow(x, z, b, self.delta, self.tau)
        
        # 债务调整项
        debt_adj = ((1 - self.kappa_b) * Qp - Q) * eta
        
        # 初始现金流
        cf0p_raw = prod + debt_adj
        
        # 股权融资成本调整（软化版本）
        equity_cost = self.kappa_e * F.relu(-cf0p_raw)
        cf0p = cf0p_raw - equity_cost * cf0p_raw.sign()
        
        return cf0p
    
    def compute_bellman_residual(
        self,
        P0: torch.Tensor,
        CF0p: torch.Tensor,
        M_list: List[torch.Tensor],
        P_children: List[torch.Tensor],
        bar_z_children: List[torch.Tensor]
    ) -> List[torch.Tensor]:
        """
        计算 Bellman 残差（支持任意分支数）
        
        loss^{(j)} = P0 - CF0p - M^{(j)} * P'^{(j)} * (1 - bar_z'^{(j)})
        
        Args:
            P0: 当期股价
            CF0p: 当期现金流
            M_list: List[torch.Tensor] - 各路径的 SDF
            P_children: List[torch.Tensor] - 各路径的未来股价
            bar_z_children: List[torch.Tensor] - 各路径的违约阈值
        
        Returns:
            residuals: List[torch.Tensor] - 各路径的残差
        """
        residuals = []
        for M, P_child, bar_z in zip(M_list, P_children, bar_z_children):
            residual = P0 - CF0p - M * P_child * (1 - bar_z)
            residuals.append(residual)
        
        return residuals
    
    def compute_bellman_residual_legacy(
        self,
        P0: torch.Tensor,
        CF0p: torch.Tensor,
        M1: torch.Tensor,
        M2: torch.Tensor,
        P2: torch.Tensor,
        P3: torch.Tensor,
        bar_z2: torch.Tensor,
        bar_z3: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        计算 Bellman 残差（旧版双路径接口）
        """
        residuals = self.compute_bellman_residual(
            P0, CF0p,
            M_list=[M1, M2],
            P_children=[P2, P3],
            bar_z_children=[bar_z2, bar_z3]
        )
        return residuals[0], residuals[1]
    
    def compute_foc_residual(
        self,
        cf0p_grad: torch.Tensor,
        M_list: List[torch.Tensor],
        P_grads: List[torch.Tensor],
        bar_z_children: List[torch.Tensor],
        eta: torch.Tensor
    ) -> List[torch.Tensor]:
        """
        计算 FOC 残差（债务选择一阶条件）- 支持任意分支数
        
        foc^{(j)} = cf_grad + M^{(j)} * P'_grad^{(j)} * (1 - bar_z'^{(j)}) * η
        """
        residuals = []
        for M, P_grad, bar_z in zip(M_list, P_grads, bar_z_children):
            foc = cf0p_grad + M * P_grad * (1 - bar_z) * eta
            residuals.append(foc)
        return residuals
    
    def compute_foc_residual_legacy(
        self,
        cf0p_grad: torch.Tensor,
        M1: torch.Tensor,
        M2: torch.Tensor,
        P2_grad: torch.Tensor,
        P3_grad: torch.Tensor,
        bar_z2: torch.Tensor,
        bar_z3: torch.Tensor,
        eta: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        计算 FOC 残差（旧版双路径接口）
        """
        residuals = self.compute_foc_residual(
            cf0p_grad,
            M_list=[M1, M2],
            P_grads=[P2_grad, P3_grad],
            bar_z_children=[bar_z2, bar_z3],
            eta=eta
        )
        return residuals[0], residuals[1]
    
    def forward(
        self,
        P0: torch.Tensor,
        inputs: torch.Tensor,
        Q: torch.Tensor,
        Qp: torch.Tensor,
        M_list: List[torch.Tensor],
        P_children: List[torch.Tensor],
        bar_z_children: List[torch.Tensor],
        foc_only: bool = False,
        bellman_only: bool = False
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        计算 P0 损失（支持任意分支数）
        
        Args:
            P0: 当前 P0 值
            inputs: 输入状态 (b, z, η, i, x, ĉf, ln Kf)
            Q: 当前债券价格
            Qp: 最优债务对应的债券价格
            M_list: List[torch.Tensor] - 各路径的 SDF
            P_children: List[torch.Tensor] - 各路径的未来股价
            bar_z_children: List[torch.Tensor] - 各路径的违约阈值
            foc_only: 是否只计算 FOC 损失
            bellman_only: 是否只计算 Bellman 损失
        
        Returns:
            total_loss: 总损失
            loss_dict: 各分量损失字典
        """
        # 提取状态变量
        b = inputs[:, 0:1]
        z = inputs[:, 1:2]
        eta = inputs[:, 2:3]
        x = inputs[:, 4:5]
        
        # 计算现金流
        CF0p = self.compute_cashflow_p0(x, z, b, Q, Qp, eta)
        
        loss_dict = {}
        total_loss = torch.tensor(0.0, device=P0.device)
        
        # Bellman 损失
        if not foc_only:
            residuals = self.compute_bellman_residual(
                P0, CF0p, M_list, P_children, bar_z_children
            )
            
            bellman_residual = compute_aio_residual(residuals, self.aio_weight)
            loss_bellman = bellman_residual.mean()
            
            # z 惩罚
            penalty_z_bellman = compute_z_penalty(
                bellman_residual, z, self.alpha_z, self.beta_z, self.z0
            )
            
            loss_dict['loss_bellman'] = loss_bellman
            loss_dict['penalty_z_bellman'] = penalty_z_bellman
            
            total_loss = total_loss + loss_bellman + penalty_z_bellman
        
        # FOC 损失（仅当 η ≠ 0 时有意义）
        if not bellman_only and eta.sum() > 0:
            # 需要计算梯度
            if inputs.requires_grad:
                # 计算对 b' 的梯度
                # 这里简化处理，实际需要对 Qp, P_children 分别求导
                foc_residual = torch.zeros_like(P0)  # placeholder
                
                loss_foc = (foc_residual.abs() * eta).mean()
                penalty_z_foc = compute_z_penalty(
                    foc_residual.abs() * eta, z, self.alpha_z, self.beta_z, self.z0
                )
                
                loss_dict['loss_foc'] = loss_foc
                loss_dict['penalty_z_foc'] = penalty_z_foc
                
                total_loss = total_loss + loss_foc + penalty_z_foc
        
        # 单调性惩罚
        if inputs.requires_grad:
            # ∂P0/∂b < 0（P0 对 b 应单调递减）
            mono_b = compute_monotonicity_penalty(P0, inputs, 0, 'negative')
            # ∂P0/∂z > 0（P0 对 z 应单调递增）
            mono_z = compute_monotonicity_penalty(P0, inputs, 1, 'positive')
            
            mono_penalty = self.lambda_b * mono_b + self.lambda_z * mono_z
            
            loss_dict['mono_penalty'] = mono_penalty
            total_loss = total_loss + mono_penalty
        
        loss_dict['total_loss'] = total_loss
        
        return total_loss, loss_dict
    
    def forward_legacy(
        self,
        P0: torch.Tensor,
        inputs: torch.Tensor,
        Q: torch.Tensor,
        Qp: torch.Tensor,
        M1: torch.Tensor,
        M2: torch.Tensor,
        P2: torch.Tensor,
        P3: torch.Tensor,
        bar_z2: torch.Tensor,
        bar_z3: torch.Tensor,
        foc_only: bool = False,
        bellman_only: bool = False
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        旧版接口（兼容2分支情况）
        """
        return self.forward(
            P0, inputs, Q, Qp,
            M_list=[M1, M2],
            P_children=[P2, P3],
            bar_z_children=[bar_z2, bar_z3],
            foc_only=foc_only,
            bellman_only=bellman_only
        )
        
        # FOC 损失（仅当 η ≠ 0 时有意义）
        if not bellman_only and eta.sum() > 0:
            # 需要计算梯度
            if inputs.requires_grad:
                # 计算对 b' 的梯度
                # 这里简化处理，实际需要对 Qp, P2, P3 分别求导
                foc_residual = torch.zeros_like(P0)  # placeholder
                
                loss_foc = (foc_residual.abs() * eta).mean()
                penalty_z_foc = compute_z_penalty(
                    foc_residual.abs() * eta, z, self.alpha_z, self.beta_z, self.z0
                )
                
                loss_dict['loss_foc'] = loss_foc
                loss_dict['penalty_z_foc'] = penalty_z_foc
                
                total_loss = total_loss + loss_foc + penalty_z_foc
        
        # 单调性惩罚
        if inputs.requires_grad:
            # ∂P0/∂b < 0（P0 对 b 应单调递减）
            mono_b = compute_monotonicity_penalty(P0, inputs, 0, 'negative')
            # ∂P0/∂z > 0（P0 对 z 应单调递增）
            mono_z = compute_monotonicity_penalty(P0, inputs, 1, 'positive')
            
            mono_penalty = self.lambda_b * mono_b + self.lambda_z * mono_z
            
            loss_dict['mono_penalty'] = mono_penalty
            total_loss = total_loss + mono_penalty
        
        loss_dict['total_loss'] = total_loss
        
        return total_loss, loss_dict
    
    def forward_simplified(
        self,
        P0: torch.Tensor,
        CF0p: torch.Tensor,
        M_list: List[torch.Tensor],
        P_children: List[torch.Tensor],
        bar_z_children: List[torch.Tensor],
        z: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        简化版 P0 损失（支持任意分支数）
        """
        # Bellman 残差
        residuals = self.compute_bellman_residual(
            P0, CF0p, M_list, P_children, bar_z_children
        )
        
        # AIO 残差
        bellman_residual = compute_aio_residual(residuals, self.aio_weight)
        loss_bellman = bellman_residual.mean()
        
        # z 惩罚
        penalty_z = compute_z_penalty(bellman_residual, z, self.alpha_z, self.beta_z, self.z0)
        
        total_loss = loss_bellman + penalty_z
        
        loss_dict = {
            'loss_bellman': loss_bellman,
            'penalty_z': penalty_z,
            'total_loss': total_loss
        }
        
        return total_loss, loss_dict
    
    def forward_simplified_legacy(
        self,
        P0: torch.Tensor,
        CF0p: torch.Tensor,
        M1: torch.Tensor,
        M2: torch.Tensor,
        P2: torch.Tensor,
        P3: torch.Tensor,
        bar_z2: torch.Tensor,
        bar_z3: torch.Tensor,
        z: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        简化版 P0 损失（旧版双路径接口）
        """
        return self.forward_simplified(
            P0, CF0p,
            M_list=[M1, M2],
            P_children=[P2, P3],
            bar_z_children=[bar_z2, bar_z3],
            z=z
        )
# --- End embedded: losses/p0_loss.py ---


In [13]:
# Ensure P0Loss exists (in case the embedded p0_loss cell wasn't run)
if 'P0Loss' not in globals():
    _p0_loss_src = '"""\nP0 Loss: 股价相关损失（不投资情形）\n\n核心目标：拟合股价 Bellman 方程残差与债务选择 FOC 残差，\n叠加单调性惩罚与 z 值惩罚，确保经济意义一致性。\n\n支持任意数量的分支路径：\n- parent: (P0_t, CF0p_t, ...) → t 期父节点\n- children: [(P_{t+1}^{(j)}, bar_z_{t+1}^{(j)}, ...)] → N 条模拟路径\n\nL_P0 = loss_bellman + loss_foc + mono_penalty\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Tuple, Dict, Optional, List\n\n\n\n\nclass P0Loss(nn.Module):\n    """\n    P0（不投资时股价）损失函数\n    \n    包含：\n    - Bellman 残差损失\n    - FOC 残差损失（债务选择一阶条件）\n    - 单调性惩罚\n    - z 值惩罚\n    """\n    \n    def __init__(\n        self,\n        delta: float = None,\n        tau: float = None,\n        kappa_b: float = None,\n        kappa_e: float = None,\n        aio_weight: float = None,\n        alpha_z: float = None,\n        beta_z: float = None,\n        z0: float = None,\n        lambda_b: float = 1.0,\n        lambda_z: float = 1.0\n    ):\n        super().__init__()\n        \n        # 经济参数\n        self.delta = delta if delta is not None else Config.DELTA\n        self.tau = tau if tau is not None else Config.TAU\n        self.kappa_b = kappa_b if kappa_b is not None else Config.KAPPA_B\n        self.kappa_e = kappa_e if kappa_e is not None else Config.KAPPA_E\n        \n        # 损失权重\n        self.aio_weight = aio_weight if aio_weight is not None else Config.AIO_WEIGHT\n        self.alpha_z = alpha_z if alpha_z is not None else Config.ALPHA_Z\n        self.beta_z = beta_z if beta_z is not None else Config.BETA_Z\n        self.z0 = z0 if z0 is not None else Config.Z0\n        self.lambda_b = lambda_b\n        self.lambda_z = lambda_z\n    \n    def compute_cashflow_p0(\n        self,\n        x: torch.Tensor,\n        z: torch.Tensor,\n        b: torch.Tensor,\n        Q: torch.Tensor,\n        Qp: torch.Tensor,\n        eta: torch.Tensor\n    ) -> torch.Tensor:\n        """\n        计算不投资时的股东现金流\n        \n        CF0p = prod + ((1-κb)Qp - Q) * η + κe * max(-CF0p, 0) * CF0p\n        """\n        # 税后利润\n        prod = compute_cashflow(x, z, b, self.delta, self.tau)\n        \n        # 债务调整项\n        debt_adj = ((1 - self.kappa_b) * Qp - Q) * eta\n        \n        # 初始现金流\n        cf0p_raw = prod + debt_adj\n        \n        # 股权融资成本调整（软化版本）\n        equity_cost = self.kappa_e * F.relu(-cf0p_raw)\n        cf0p = cf0p_raw - equity_cost * cf0p_raw.sign()\n        \n        return cf0p\n    \n    def compute_bellman_residual(\n        self,\n        P0: torch.Tensor,\n        CF0p: torch.Tensor,\n        M_list: List[torch.Tensor],\n        P_children: List[torch.Tensor],\n        bar_z_children: List[torch.Tensor]\n    ) -> List[torch.Tensor]:\n        """\n        计算 Bellman 残差（支持任意分支数）\n        \n        loss^{(j)} = P0 - CF0p - M^{(j)} * P\'^{(j)} * (1 - bar_z\'^{(j)})\n        \n        Args:\n            P0: 当期股价\n            CF0p: 当期现金流\n            M_list: List[torch.Tensor] - 各路径的 SDF\n            P_children: List[torch.Tensor] - 各路径的未来股价\n            bar_z_children: List[torch.Tensor] - 各路径的违约阈值\n        \n        Returns:\n            residuals: List[torch.Tensor] - 各路径的残差\n        """\n        residuals = []\n        for M, P_child, bar_z in zip(M_list, P_children, bar_z_children):\n            residual = P0 - CF0p - M * P_child * (1 - bar_z)\n            residuals.append(residual)\n        \n        return residuals\n    \n    def compute_bellman_residual_legacy(\n        self,\n        P0: torch.Tensor,\n        CF0p: torch.Tensor,\n        M1: torch.Tensor,\n        M2: torch.Tensor,\n        P2: torch.Tensor,\n        P3: torch.Tensor,\n        bar_z2: torch.Tensor,\n        bar_z3: torch.Tensor\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        """\n        计算 Bellman 残差（旧版双路径接口）\n        """\n        residuals = self.compute_bellman_residual(\n            P0, CF0p,\n            M_list=[M1, M2],\n            P_children=[P2, P3],\n            bar_z_children=[bar_z2, bar_z3]\n        )\n        return residuals[0], residuals[1]\n    \n    def compute_foc_residual(\n        self,\n        cf0p_grad: torch.Tensor,\n        M_list: List[torch.Tensor],\n        P_grads: List[torch.Tensor],\n        bar_z_children: List[torch.Tensor],\n        eta: torch.Tensor\n    ) -> List[torch.Tensor]:\n        """\n        计算 FOC 残差（债务选择一阶条件）- 支持任意分支数\n        \n        foc^{(j)} = cf_grad + M^{(j)} * P\'_grad^{(j)} * (1 - bar_z\'^{(j)}) * η\n        """\n        residuals = []\n        for M, P_grad, bar_z in zip(M_list, P_grads, bar_z_children):\n            foc = cf0p_grad + M * P_grad * (1 - bar_z) * eta\n            residuals.append(foc)\n        return residuals\n    \n    def compute_foc_residual_legacy(\n        self,\n        cf0p_grad: torch.Tensor,\n        M1: torch.Tensor,\n        M2: torch.Tensor,\n        P2_grad: torch.Tensor,\n        P3_grad: torch.Tensor,\n        bar_z2: torch.Tensor,\n        bar_z3: torch.Tensor,\n        eta: torch.Tensor\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        """\n        计算 FOC 残差（旧版双路径接口）\n        """\n        residuals = self.compute_foc_residual(\n            cf0p_grad,\n            M_list=[M1, M2],\n            P_grads=[P2_grad, P3_grad],\n            bar_z_children=[bar_z2, bar_z3],\n            eta=eta\n        )\n        return residuals[0], residuals[1]\n    \n    def forward(\n        self,\n        P0: torch.Tensor,\n        inputs: torch.Tensor,\n        Q: torch.Tensor,\n        Qp: torch.Tensor,\n        M_list: List[torch.Tensor],\n        P_children: List[torch.Tensor],\n        bar_z_children: List[torch.Tensor],\n        foc_only: bool = False,\n        bellman_only: bool = False\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        """\n        计算 P0 损失（支持任意分支数）\n        \n        Args:\n            P0: 当前 P0 值\n            inputs: 输入状态 (b, z, η, i, x, ĉf, ln Kf)\n            Q: 当前债券价格\n            Qp: 最优债务对应的债券价格\n            M_list: List[torch.Tensor] - 各路径的 SDF\n            P_children: List[torch.Tensor] - 各路径的未来股价\n            bar_z_children: List[torch.Tensor] - 各路径的违约阈值\n            foc_only: 是否只计算 FOC 损失\n            bellman_only: 是否只计算 Bellman 损失\n        \n        Returns:\n            total_loss: 总损失\n            loss_dict: 各分量损失字典\n        """\n        # 提取状态变量\n        b = inputs[:, 0:1]\n        z = inputs[:, 1:2]\n        eta = inputs[:, 2:3]\n        x = inputs[:, 4:5]\n        \n        # 计算现金流\n        CF0p = self.compute_cashflow_p0(x, z, b, Q, Qp, eta)\n        \n        loss_dict = {}\n        total_loss = torch.tensor(0.0, device=P0.device)\n        \n        # Bellman 损失\n        if not foc_only:\n            residuals = self.compute_bellman_residual(\n                P0, CF0p, M_list, P_children, bar_z_children\n            )\n            \n            bellman_residual = compute_aio_residual(residuals, self.aio_weight)\n            loss_bellman = bellman_residual.mean()\n            \n            # z 惩罚\n            penalty_z_bellman = compute_z_penalty(\n                bellman_residual, z, self.alpha_z, self.beta_z, self.z0\n            )\n            \n            loss_dict[\'loss_bellman\'] = loss_bellman\n            loss_dict[\'penalty_z_bellman\'] = penalty_z_bellman\n            \n            total_loss = total_loss + loss_bellman + penalty_z_bellman\n        \n        # FOC 损失（仅当 η ≠ 0 时有意义）\n        if not bellman_only and eta.sum() > 0:\n            # 需要计算梯度\n            if inputs.requires_grad:\n                # 计算对 b\' 的梯度\n                # 这里简化处理，实际需要对 Qp, P_children 分别求导\n                foc_residual = torch.zeros_like(P0)  # placeholder\n                \n                loss_foc = (foc_residual.abs() * eta).mean()\n                penalty_z_foc = compute_z_penalty(\n                    foc_residual.abs() * eta, z, self.alpha_z, self.beta_z, self.z0\n                )\n                \n                loss_dict[\'loss_foc\'] = loss_foc\n                loss_dict[\'penalty_z_foc\'] = penalty_z_foc\n                \n                total_loss = total_loss + loss_foc + penalty_z_foc\n        \n        # 单调性惩罚\n        if inputs.requires_grad:\n            # ∂P0/∂b < 0（P0 对 b 应单调递减）\n            mono_b = compute_monotonicity_penalty(P0, inputs, 0, \'negative\')\n            # ∂P0/∂z > 0（P0 对 z 应单调递增）\n            mono_z = compute_monotonicity_penalty(P0, inputs, 1, \'positive\')\n            \n            mono_penalty = self.lambda_b * mono_b + self.lambda_z * mono_z\n            \n            loss_dict[\'mono_penalty\'] = mono_penalty\n            total_loss = total_loss + mono_penalty\n        \n        loss_dict[\'total_loss\'] = total_loss\n        \n        return total_loss, loss_dict\n    \n    def forward_legacy(\n        self,\n        P0: torch.Tensor,\n        inputs: torch.Tensor,\n        Q: torch.Tensor,\n        Qp: torch.Tensor,\n        M1: torch.Tensor,\n        M2: torch.Tensor,\n        P2: torch.Tensor,\n        P3: torch.Tensor,\n        bar_z2: torch.Tensor,\n        bar_z3: torch.Tensor,\n        foc_only: bool = False,\n        bellman_only: bool = False\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        """\n        旧版接口（兼容2分支情况）\n        """\n        return self.forward(\n            P0, inputs, Q, Qp,\n            M_list=[M1, M2],\n            P_children=[P2, P3],\n            bar_z_children=[bar_z2, bar_z3],\n            foc_only=foc_only,\n            bellman_only=bellman_only\n        )\n        \n        # FOC 损失（仅当 η ≠ 0 时有意义）\n        if not bellman_only and eta.sum() > 0:\n            # 需要计算梯度\n            if inputs.requires_grad:\n                # 计算对 b\' 的梯度\n                # 这里简化处理，实际需要对 Qp, P2, P3 分别求导\n                foc_residual = torch.zeros_like(P0)  # placeholder\n                \n                loss_foc = (foc_residual.abs() * eta).mean()\n                penalty_z_foc = compute_z_penalty(\n                    foc_residual.abs() * eta, z, self.alpha_z, self.beta_z, self.z0\n                )\n                \n                loss_dict[\'loss_foc\'] = loss_foc\n                loss_dict[\'penalty_z_foc\'] = penalty_z_foc\n                \n                total_loss = total_loss + loss_foc + penalty_z_foc\n        \n        # 单调性惩罚\n        if inputs.requires_grad:\n            # ∂P0/∂b < 0（P0 对 b 应单调递减）\n            mono_b = compute_monotonicity_penalty(P0, inputs, 0, \'negative\')\n            # ∂P0/∂z > 0（P0 对 z 应单调递增）\n            mono_z = compute_monotonicity_penalty(P0, inputs, 1, \'positive\')\n            \n            mono_penalty = self.lambda_b * mono_b + self.lambda_z * mono_z\n            \n            loss_dict[\'mono_penalty\'] = mono_penalty\n            total_loss = total_loss + mono_penalty\n        \n        loss_dict[\'total_loss\'] = total_loss\n        \n        return total_loss, loss_dict\n    \n    def forward_simplified(\n        self,\n        P0: torch.Tensor,\n        CF0p: torch.Tensor,\n        M_list: List[torch.Tensor],\n        P_children: List[torch.Tensor],\n        bar_z_children: List[torch.Tensor],\n        z: torch.Tensor\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        """\n        简化版 P0 损失（支持任意分支数）\n        """\n        # Bellman 残差\n        residuals = self.compute_bellman_residual(\n            P0, CF0p, M_list, P_children, bar_z_children\n        )\n        \n        # AIO 残差\n        bellman_residual = compute_aio_residual(residuals, self.aio_weight)\n        loss_bellman = bellman_residual.mean()\n        \n        # z 惩罚\n        penalty_z = compute_z_penalty(bellman_residual, z, self.alpha_z, self.beta_z, self.z0)\n        \n        total_loss = loss_bellman + penalty_z\n        \n        loss_dict = {\n            \'loss_bellman\': loss_bellman,\n            \'penalty_z\': penalty_z,\n            \'total_loss\': total_loss\n        }\n        \n        return total_loss, loss_dict\n    \n    def forward_simplified_legacy(\n        self,\n        P0: torch.Tensor,\n        CF0p: torch.Tensor,\n        M1: torch.Tensor,\n        M2: torch.Tensor,\n        P2: torch.Tensor,\n        P3: torch.Tensor,\n        bar_z2: torch.Tensor,\n        bar_z3: torch.Tensor,\n        z: torch.Tensor\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        """\n        简化版 P0 损失（旧版双路径接口）\n        """\n        return self.forward_simplified(\n            P0, CF0p,\n            M_list=[M1, M2],\n            P_children=[P2, P3],\n            bar_z_children=[bar_z2, bar_z3],\n            z=z\n        )'
    exec(_p0_loss_src, globals())
    del _p0_loss_src


In [14]:
import torch

torch.manual_seed(0)
device = torch.device('cpu')


In [15]:
# 1) Random batch with shape (batch, 3, feat) for SDF/FC1
batch_size = 8
feat = 7  # [b, z, ETA, i, x, Hatcf, LnKF]
nodes = torch.randn(batch_size, 3, feat, device=device)

parent = nodes[:, 0, :]
children = nodes[:, 1:, :]  # (batch, 2, feat)

print('nodes:', nodes.shape)
print('parent:', parent.shape)
print('children:', children.shape)


nodes: torch.Size([8, 3, 7])
parent: torch.Size([8, 7])
children: torch.Size([8, 2, 7])


In [16]:
# 2) SDF/FC1 forward + loss
model = SDFFC1Combined().to(device)
loss_fn = SDFLoss().to(device)

w_parent, w_children, M, c_children, k_children = model.forward_step(
    x_prev=parent[:, 4:5],
    x_curr=children[:, :, 4:5],
    hatcf_prev=parent[:, 5:6],
    lnkf_prev=parent[:, 6:7],
    return_physical=True
)

c_parent = parent[:, 5:6]
k_parent = parent[:, 6:7]

residuals = loss_fn.compute_euler_residuals(
    w_parent.squeeze(-1), w_children.squeeze(-1),
    k_parent.squeeze(-1), k_children.squeeze(-1),
    c_parent.squeeze(-1), c_children.squeeze(-1)
)  # (batch, 2)
combined = residuals.prod(dim=-1)
main_loss = torch.log1p(combined.abs()).mean()

total_loss = loss_fn(
    w_parent=w_parent.squeeze(-1),
    w_children=w_children.squeeze(-1),
    M_list=M.squeeze(-1) if M.dim() == 3 else M,
    k_parent=k_parent.squeeze(-1),
    k_children=k_children.squeeze(-1),
    c_parent=c_parent.squeeze(-1),
    c_children=c_children.squeeze(-1),
)

print('residuals:', residuals.shape)
print('combined:', combined.shape)
print('main_loss:', main_loss.item())
print('total_loss:', total_loss.item())


residuals: torch.Size([8, 2])
combined: torch.Size([8])
main_loss: inf
total_loss: inf


In [17]:
# 3) Policy/Value: test P_children and Bellman residuals (stop at main_loss)
pv_model = PolicyValueModel().to(device)
p0_loss_fn = P0Loss().to(device)

parent_pv = torch.randn(batch_size, 7, device=device)
children_pv = torch.randn(batch_size, 2, 7, device=device)

output_t = pv_model(parent_pv)

def _get_out(out, name: str, idx: int):
    if isinstance(out, dict):
        return out[name]
    if hasattr(out, name):
        return getattr(out, name)
    return out[:, idx:idx + 1]

bp0_t = _get_out(output_t, 'bp0', 1)
bpI_t = _get_out(output_t, 'bpI', 2)
bar_i_t = _get_out(output_t, 'bar_i', 4)
bp_t = _get_out(output_t, 'bp', -1)
if bp_t.shape != bp0_t.shape:
    bp_t = bar_i_t * bpI_t + (1 - bar_i_t) * bp0_t
b_parent = parent_pv[:, 0:1]

output_children = []
for i in range(children_pv.shape[1]):
    child = children_pv[:, i, :]
    eta_child = child[:, 2:3]
    child_state = child.clone()
    child_state[:, 0:1] = eta_child * bp_t + (1 - eta_child) * b_parent
    output_children.append(pv_model(child_state))

childp0_state = parent_pv.clone()
childp0_state[:, 0:1] = bpI_t
outputp0_children = pv_model(childp0_state)

P0 = _get_out(output_t, 'P0', 3)
P_children = [_get_out(out, 'P', 7) for out in output_children]
bar_z_children = [_get_out(out, 'bar_z', 6) for out in output_children]

Q = _get_out(output_t, 'Q', 0)
Qp = _get_out(outputp0_children, 'Q', 0)

CF0p = p0_loss_fn.compute_cashflow_p0(
    parent_pv[:, 4:5],
    parent_pv[:, 1:2],
    parent_pv[:, 0:1],
    Q, Qp,
    parent_pv[:, 2:3]
)

M_list = [torch.ones(batch_size, 1, device=device) for _ in P_children]
residuals = p0_loss_fn.compute_bellman_residual(
    P0, CF0p, M_list, P_children, bar_z_children
)
residuals_stack = torch.stack(residuals, dim=-1)
combined = residuals_stack.prod(dim=-1)
main_loss = combined.abs().mean()

print('parent_pv:', parent_pv.shape)
print('P_children shapes:', [p.shape for p in P_children])
print('residuals_stack:', residuals_stack.shape)
print('combined:', combined.shape)
print('main_loss:', main_loss.item())


parent_pv: torch.Size([8, 7])
P_children shapes: [torch.Size([8, 1]), torch.Size([8, 1])]
residuals_stack: torch.Size([8, 1, 2])
combined: torch.Size([8, 1])
main_loss: 5.25921630859375


In [18]:
residuals_stack

tensor([[[-1.3490, -1.4180]],

        [[-4.4148, -4.4816]],

        [[ 0.3787,  0.3481]],

        [[-2.8183, -2.8290]],

        [[-1.2378, -1.2803]],

        [[-2.0728, -1.9199]],

        [[-2.3791, -2.3768]],

        [[ 0.9905,  1.0618]]], grad_fn=<StackBackward0>)

In [19]:
residuals

[tensor([[-1.3490],
         [-4.4148],
         [ 0.3787],
         [-2.8183],
         [-1.2378],
         [-2.0728],
         [-2.3791],
         [ 0.9905]], grad_fn=<SubBackward0>),
 tensor([[-1.4180],
         [-4.4816],
         [ 0.3481],
         [-2.8290],
         [-1.2803],
         [-1.9199],
         [-2.3768],
         [ 1.0618]], grad_fn=<SubBackward0>)]